In [2]:
import re
import warnings
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize, LabelEncoder

import skfuzzy as fuzz

import nltk
from nltk.stem import WordNetLemmatizer

from preprocessing_utils import preprocess_corpus
from metrics_utils import *

seed = 42
np.random.seed(seed)

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/bernardod/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/bernardod/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/bernardod/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to /home/bernardod/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
df = pd.read_json("../datasets/fixed_dataset.json")

In [4]:
df.shape

(121, 6)

In [5]:
def get_first_label(category: str) -> str:
    return str(category).split(",")[0].strip()

df["category"] = df["category"].apply(get_first_label)
label_encoder = LabelEncoder()
y_true_encoded = label_encoder.fit_transform(df["category"].apply(get_first_label))

df["title_abstract"] = df["title"] + " " + df["abstract"] 

In [6]:
print(f"Distribuition by label: {np.bincount(y_true_encoded)}")

Distribuition by label: [ 5  8 21 21 12  5 41  8]


In [7]:
X_preprocessed = preprocess_corpus(
    df["title_abstract"], min_df=3, max_df=0.7, max_features=3000, ngram_range=(1, 3)
)

Starting lemmatization and cleaning...
Starting vectorization (Tokenization, Stopwords, Pruning)...


In [8]:
N_COMPONENTS_VALUES = [20, 25, 35, 50, 75]
K_VALUES = [2, 3, 4, 6, 8, 10, 12, 15]

In [9]:
def fuzzy_cmeans_fit(X, n_clusters, m=1.7, error=0.005, maxiter=1000, random_state=seed):
    cntr, U, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
        X.T,
        c=n_clusters,
        m=m,
        error=error,
        maxiter=maxiter,
        metric="cosine",
        seed=random_state
    )

    return U.T, float(fpc)

def diagnose_collapse(U):
    """Heuristic collapse flag based on high normalized entropy."""
    eps = 1e-12
    n_clusters = U.shape[1]
    entropy_per_sample = -np.sum(U * np.log(U + eps), axis=1)
    max_entropy = np.log(n_clusters)
    normalized_entropy = float(np.mean(entropy_per_sample) / max_entropy)
    avg_max_membership = float(U.max(axis=1).mean())

    return {
        "collapsed": normalized_entropy > 0.85,
        "entropy_norm": normalized_entropy,
        "avg_max_memb": avg_max_membership,
    }

def evaluate(X, y_true, y_pred):
    return {
        "ARI": calculate_ari(y_true, y_pred),
        "NMI": calculate_nmi(y_true, y_pred),
        "ACC": calculate_accuracy(y_true, y_pred),
        "SIL": calculate_silhouette(X, y_pred)
    }

In [10]:
rows = []

for n_comp in N_COMPONENTS_VALUES:
    svd = TruncatedSVD(n_components=n_comp, random_state=seed)
    X = svd.fit_transform(X_preprocessed)
    X = normalize(X, norm="l2")
    for K in K_VALUES:
        U, fpc = fuzzy_cmeans_fit(X, n_clusters=K)
        y_pred = U.argmax(axis=1)
        is_collapsed = diagnose_collapse(U)
        metrics = evaluate(X=X, y_true=y_true_encoded, y_pred=y_pred)

        row = {
            "n_comp": n_comp,
            "K": K,
            "ARI": metrics["ARI"],
            "NMI": metrics["NMI"],
            "ACC": metrics["ACC"],
            "SIL": metrics["SIL"],
            "collapsed": is_collapsed["collapsed"],
            "entropy_norm": is_collapsed["entropy_norm"],
            "avg_max_memb": is_collapsed["avg_max_memb"],
            "fpc": fpc,

        }
        rows.append(row)

results_df = pd.DataFrame(rows)
results_df.sort_values(by=["ARI", "NMI", "ACC"], ascending=[False, False, False], inplace=True)
results_df.reset_index(drop=True, inplace=True)

In [11]:
topN = (
    results_df[~results_df["collapsed"]]
    .sort_values(["NMI", "ARI", "ACC"], ascending=False)
    .head(20)
)
topN

,n_comp,K,ARI,NMI,ACC,SIL,collapsed,entropy_norm,avg_max_memb,fpc
12,20,15,0.043292,0.280184,0.289256,0.259738,False,0.491371,0.621235,0.514761
24,25,15,0.028368,0.260114,0.247934,0.225871,False,0.571186,0.551873,0.444901
15,20,12,0.039689,0.250193,0.264463,0.201955,False,0.562445,0.574699,0.462845
16,25,12,0.039341,0.242567,0.256198,0.187916,False,0.604298,0.538345,0.429358
21,20,10,0.031661,0.229949,0.280992,0.184276,False,0.593828,0.559755,0.449835
17,25,10,0.038933,0.225681,0.297521,0.146534,False,0.653524,0.505151,0.398645
13,20,8,0.042280,0.202948,0.297521,0.171989,False,0.640365,0.550019,0.429979
7,20,6,0.051982,0.196727,0.322314,0.160557,False,0.668647,0.562267,0.443049
26,25,8,0.024334,0.190649,0.272727,0.146584,False,0.684521,0.505017,0.394262
28,25,6,0.021572,0.155950,0.280992,0.138775,False,0.734344,0.508777,0.387888


#### Best result at the moment 

##### preprocess_corpus(df["title_abstract"], min_df=3, max_df=0.7, max_features=3000, ngram_range=(1, 3))
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>n_comp</th>
      <th>K</th>
      <th>ARI</th>
      <th>NMI</th>
      <th>ACC</th>
      <th>SIL</th>
      <th>collapsed</th>
      <th>entropy_norm</th>
      <th>avg_max_memb</th>
      <th>fpc</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>12</th>
      <td>20</td>
      <td>15</td>
      <td>0.043292</td>
      <td>0.280184</td>
      <td>0.289256</td>
      <td>0.259738</td>
      <td>False</td>
      <td>0.491371</td>
      <td>0.621235</td>
      <td>0.514761</td>
    </tr>
    <tr>
      <th>24</th>
      <td>25</td>
      <td>15</td>
      <td>0.028368</td>
      <td>0.260114</td>
      <td>0.247934</td>
      <td>0.225871</td>
      <td>False</td>
      <td>0.571186</td>
      <td>0.551873</td>
      <td>0.444901</td>
    </tr>
    <tr>
      <th>15</th>
      <td>20</td>
      <td>12</td>
      <td>0.039689</td>
      <td>0.250193</td>
      <td>0.264463</td>
      <td>0.201955</td>
      <td>False</td>
      <td>0.562445</td>
      <td>0.574699</td>
      <td>0.462845</td>
    </tr>
    <tr>
      <th>16</th>
      <td>25</td>
      <td>12</td>
      <td>0.039341</td>
      <td>0.242567</td>
      <td>0.256198</td>
      <td>0.187916</td>
      <td>False</td>
      <td>0.604298</td>
      <td>0.538345</td>
      <td>0.429358</td>
    </tr>
    <tr>
      <th>21</th>
      <td>20</td>
      <td>10</td>
      <td>0.031661</td>
      <td>0.229949</td>
      <td>0.280992</td>
      <td>0.184276</td>
      <td>False</td>
      <td>0.593828</td>
      <td>0.559755</td>
      <td>0.449835</td>
    </tr>
  </tbody>
</table>
</div>